# WiC: binary word sense disambiguation

[WiC](https://pilehvar.github.io/wic/) gives a target word in two sentences. The task: does the
word have the same sense in both? Guessing scores 50%.

Every method gives each pair one cosine similarity. A threshold θ, fitted on the train split,
turns that into yes or no.

The rows in the results:

- **static** (Tucker factor, GloVe, word2vec): the average of the word vectors around the target.
- **contextual** (Tucker): the target's vector inside its n-gram windows.
- **excluded** (Tucker): what the neighbouring words predict at the target's place, without
  reading the target itself.

## 0. Setup

In [15]:
# Pin the GPUs before torch touches CUDA.
from tensormet.utils import select_gpu
DEVICE = select_gpu([1, 2])

import sys, json
from pathlib import Path

import numpy as np

# Anchor to the repo root, as in the SPINE notebook. wic_eval.py sits next to this
# notebook; glove_baseline.py lives in 5_evaluation/.
def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / '5_evaluation' / 'third_party').is_dir() and (p / 'src' / 'tensormet').is_dir():
            return p
    raise RuntimeError(f'repo root not found above {start}')

REPO = _find_repo_root(Path.cwd())
EVAL_DIR = REPO / '5_evaluation'
NB_DIR = EVAL_DIR / 'wsd'

for d in (NB_DIR, EVAL_DIR):
    if str(d) not in sys.path:
        sys.path.insert(0, str(d))

import wic_eval as we
from glove_baseline import load_glove, load_word2vec
from tensormet.tucker_tensor import TuckerDecomposition
from tensormet.experimental.contextual import ContextualEncoder, cosine
from tensormet.utils import DATA_DIR

OUT = NB_DIR / 'out'; OUT.mkdir(exist_ok=True)
RESULTS = OUT / 'results.jsonl'

WIC = we.download_wic(NB_DIR / 'data')
print('WiC data under', WIC)

Pinned to GPU(s): [1, 2]
WiC data under /home/pricie/stefa/pc/5_evaluation/wsd/data


In [16]:
# Expect 5428 / 638 / 1400 items and roughly half T. A split without gold is
# loaded but not scored.
checks = we.preflight(WIC)
data = {s: we.load_wic(WIC, s) for s in we.SPLITS if checks[s]['ok']}
gold = we.labels(data)
print('scored splits:', list(gold))

     ok  train  5428 items  gold, 50.0% T
     ok  dev     638 items  gold, 50.0% T
     ok  test   1400 items  gold, 50.0% T
scored splits: ['train', 'dev', 'test']


## 1. Load the model

In [29]:
tk = TuckerDecomposition.load_from_disk(dataset="6-gram-raw-bos-eos-fineweb-en_1B", decomposition="tt", rank=300)
enc = ContextualEncoder(tk, device=DEVICE)
print(enc.spec, '| device', enc.device, '| rank', enc.rank, '| vocab', len(enc.word_index))

NgramSpec(order=6, raw=True, padded=True, spacy_model='en_core_web_md') | device cuda:0 | rank 300 | vocab 10000


## 2. Baselines

Pretrained GloVe and word2vec as downloaded, cut down to the words our model also has.

In [18]:
# GloVe 2024 (wiki+gigaword) at the model's width: the release ships 50/100/200/300d.
_glove_files = sorted((DATA_DIR / 'corpora').glob(f'wiki_giga_2024_{enc.rank}_*.txt'))
if not _glove_files:
    raise FileNotFoundError(f'no {enc.rank}d wiki_giga_2024 GloVe in {DATA_DIR / "corpora"}; '
                            f'have {sorted(p.name for p in (DATA_DIR / "corpora").glob("wiki_giga_2024_*"))}')
GLOVE = _glove_files[0]

import gensim.downloader
W2V_KV = gensim.downloader.load('word2vec-google-news-300')  # also used by the fixed block (6b)

glove = load_glove(GLOVE, like=tk)
w2v = load_word2vec(like=tk, keyed_vectors=W2V_KV)

def _words(model):
    return set(model.vocab[f'vocab_{model.roles[0]}'])

# Sentence-boundary and placeholder entries are not words.
SPECIAL = {'<s>', '</s>', '<BOS>', '<EOS>', '~', ''}
EVAL_VOCAB = (_words(tk) - SPECIAL) & _words(glove) & _words(w2v)
print(len(EVAL_VOCAB), 'words shared by all three')

TABLES = {
    'tucker (factor)': we.StaticTable(tk, EVAL_VOCAB),
    f'glove-{glove.get_rank()}d': we.StaticTable(glove, EVAL_VOCAB),
    f'word2vec-{w2v.get_rank()}d': we.StaticTable(w2v, EVAL_VOCAB),
}

9987 vectors of dim 300 read from wiki_giga_2024_300_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt (13 of the model's words missing)
9789 vectors of dim 300 read from word2vec-google-news-300 (211 of the model's words missing)
9784 words shared by all three


## 3. Coverage

How many pairs each method can score. Our vocabulary has 10k words, so many WiC targets are
missing from it.

In [30]:
slots = {s: we.locate_targets(items, enc.spec) for s, items in data.items()}

for s, items in data.items():
    bad = we.target_mismatches(items, slots[s])
    print(f'{s:<5} {len(bad)} of {2 * len(items)} target tokens differ from the WiC word;', bad[:5])

train 0 of 10856 target tokens differ from the WiC word; []
dev   0 of 1276 target tokens differ from the WiC word; []
test  1 of 2800 target tokens differ from the WiC word; [('have', "'ve", None)]


In [31]:
import pandas as pd

ctx_ok = {s: we.contextual_similarities(enc, slots[s], rounds=1)[1] for s in data}
cov = {s: we.coverage(data[s], slots[s], EVAL_VOCAB, ctx_ok[s]) for s in data}
(OUT / 'coverage.json').write_text(json.dumps(cov, indent=2), encoding='utf-8')

# Per split: the share of pairs with both targets in the vocabulary, the share with a
# contextual vector for both, and the share of the other words that are in the vocabulary.
pd.DataFrame.from_dict({s: {'pairs': c['items'],
                            'target in vocabulary': c['target_in_vocab_both'],
                            'contextual vector': c['contextual_vector_both'],
                            'neighbours in vocabulary': c['context_token_coverage']}
                        for s, c in cov.items()}, orient='index')

,pairs,target in vocabulary,contextual vector,neighbours in vocabulary
train,5428,0.8749,0.7697,0.9219
dev,638,0.5956,0.4984,0.9120
test,1400,0.6200,0.5436,0.9177


In [32]:
# Why the contextual encoder skips a pair, counted per sentence.
# Full lists: out/unscored_<split>.csv
from collections import Counter

for s in data:
    df = we.unscored(data[s], slots[s], ctx_ok[s], set(enc.word_index), order=enc.spec.order)
    df.to_csv(OUT / f'unscored_{s}.csv')
    reasons = Counter(df['reason 1']) + Counter(df['reason 2'])
    reasons.pop('ok', None)
    print(f'{s:<5} {len(df)}/{len(data[s])} pairs skipped: {dict(reasons.most_common())}')

train 1250/5428 pairs skipped: {'target OOV': 957, 'no in-vocab window': 615, 'target OOV, lemma in vocab': 195}
dev   320/638 pairs skipped: {'target OOV': 399, 'no in-vocab window': 68, 'target OOV, lemma in vocab': 59}
test  639/1400 pairs skipped: {'target OOV': 851, 'no in-vocab window': 114, 'target OOV, lemma in vocab': 104, 'not located': 1}


## 4. Static vectors

The average of the word vectors within ±`window` words of the target (target included).
`sentence` is the whole sentence.

In [33]:
WINDOWS = (0, 1, None)
SIMS = {}
for name, table in TABLES.items():
    for w in WINDOWS:
        label = f"{name} | static | window={'sentence' if w is None else w}"
        SIMS[label] = {s: we.static_similarities(table, slots[s], w) for s in data}
        print(f'{label:<45}', {s: f'{int(ok.sum())}/{len(ok)}' for s, (_, ok) in SIMS[label].items()})

tucker (factor) | static | window=0           {'train': '4749/5428', 'dev': '380/638', 'test': '868/1400'}
tucker (factor) | static | window=1           {'train': '5389/5428', 'dev': '626/638', 'test': '1372/1400'}
tucker (factor) | static | window=sentence    {'train': '5417/5428', 'dev': '633/638', 'test': '1394/1400'}
glove-300d | static | window=0                {'train': '4749/5428', 'dev': '380/638', 'test': '868/1400'}
glove-300d | static | window=1                {'train': '5389/5428', 'dev': '626/638', 'test': '1372/1400'}
glove-300d | static | window=sentence         {'train': '5417/5428', 'dev': '633/638', 'test': '1394/1400'}
word2vec-300d | static | window=0             {'train': '4749/5428', 'dev': '380/638', 'test': '868/1400'}
word2vec-300d | static | window=1             {'train': '5389/5428', 'dev': '626/638', 'test': '1372/1400'}
word2vec-300d | static | window=sentence      {'train': '5417/5428', 'dev': '633/638', 'test': '1394/1400'}


## 5. Contextual vectors (Tucker)

`rounds=1` uses the n-gram windows that contain the target. `rounds=3` also brings in context
from further away.

In [34]:
for rounds in (1,3):
    label = f"tucker | contextual | rounds={rounds}"
    SIMS[label] = {s: we.contextual_similarities(enc, slots[s], rounds=rounds) for s in data}
    print(f'{label:<45}', {s: f'{int(ok.sum())}/{len(ok)}' for s, (_, ok) in SIMS[label].items()})

tucker | contextual | rounds=1                {'train': '4178/5428', 'dev': '318/638', 'test': '761/1400'}
tucker | contextual | rounds=3                {'train': '4178/5428', 'dev': '318/638', 'test': '761/1400'}


### 5b. Excluded vectors (Tucker)

What the other words of each window predict at the target's place. The target itself is not
read, so out-of-vocabulary targets can be scored too.

In [35]:
label = 'tucker | excluded'
SIMS[label] = {s: we.excluded_similarities(enc, slots[s]) for s in data}
print(f'{label:<45}', {s: f'{int(ok.sum())}/{len(ok)}' for s, (_, ok) in SIMS[label].items()})

tucker | excluded                             {'train': '4751/5428', 'dev': '532/638', 'test': '1211/1400'}


## 6. Results

- **(common)**: only the pairs every method can score. This is the fair comparison.
- **(all)**: all pairs. A pair a method cannot score counts as a guess.
- **scored**: how many pairs the method could score.
- **θ**: the threshold used for the (all) columns.

Our methods:
- Excluded: contraction of the other elements of the $n$-gram over the core = core-informed prediction of the element
- Contextual: Excluded, gated by the actual vector of the target word. Due to the sparsity, this seems to throw away quite a lot of information.

$\Rightarrow$ We do better when considering only our vocab. The signal from the tucker core exists, but is not super strong.(It needs more tinkering, which we could do in future work).
Mind that the window=0 elements have a $\theta$ of 1 or -inf: they are random chance baselines. The $\theta$ (threshold cosine distance to say "different meaning") is learned from the dev set.

In [36]:
META = {'tucker': str(tk.decomp_path), 'glove': glove.source, 'word2vec': w2v.source}
rows = we.run_methods(SIMS, gold, results_path=RESULTS, meta=META)

common = we.common_mask(SIMS)
print('common subset:', {s: f'{int(m.sum())}/{len(m)}' for s, m in common.items()})

# Same inputs, same θ and accuracies (no randomness anywhere).
assert we.run_methods(SIMS, gold, meta=META) == rows

results_df = we.results_frame(rows).round(3)

# Contextual rounds 0 = baseline

results_df

common subset: {'train': '4174/5428', 'dev': '318/638', 'test': '761/1400'}


,dev (common),test (common),dev (all),test (all),dev scored,test scored,θ (all)
method,,,,,,,
tucker (factor) | static | window=0,0.500,0.498,0.505,0.505,380/638,868/1400,1.000
tucker (factor) | static | window=1,0.538,0.582,0.533,0.560,626/638,1372/1400,0.057
tucker (factor) | static | window=sentence,0.494,0.540,0.500,0.551,633/638,1394/1400,0.060
glove-300d | static | window=0,0.497,0.528,0.516,0.530,380/638,868/1400,inf
glove-300d | static | window=1,0.535,0.562,0.525,0.531,626/638,1372/1400,0.893
glove-300d | static | window=sentence,0.513,0.565,0.517,0.548,633/638,1394/1400,0.857
word2vec-300d | static | window=0,0.497,0.528,0.516,0.530,380/638,868/1400,inf
word2vec-300d | static | window=1,0.547,0.569,0.533,0.534,626/638,1372/1400,0.681
word2vec-300d | static | window=sentence,0.538,0.565,0.541,0.551,633/638,1394/1400,0.571


### 6b. Baselines with their full vocabulary

GloVe and word2vec again, but not cut down to our 10k words. These numbers do not depend on the
loaded model. Here (common) means the pairs all rows of *this* table can score.

In [26]:
from tensormet.experimental.contextual import NgramSpec

# Nothing below reads tk, enc or the model's vocabulary.
FIXED_SPEC = NgramSpec(order=4, raw=True, padded=False)  # tokenisation reads only `raw`
GLOVE_FIXED = DATA_DIR / 'corpora' / 'wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt'

fixed_slots = {s: we.locate_targets(items, FIXED_SPEC) for s, items in data.items()}
# Every word form in the WiC sentences: loading just these equals loading the full vocabulary.
WIC_WORDS = sorted({t for pairs in fixed_slots.values() for pair in pairs
                    for slot in pair if slot is not None for t in slot[0].tokens})

glove_full = load_glove(GLOVE_FIXED, roles=['w'], vocab=WIC_WORDS)
w2v_full = load_word2vec(roles=['w'], vocab=WIC_WORDS, keyed_vectors=W2V_KV)

FIXED_TABLES = {
    f'fixed | glove-{glove_full.get_rank()}d full vocab': we.StaticTable(glove_full),
    f'fixed | word2vec-{w2v_full.get_rank()}d full vocab': we.StaticTable(w2v_full),
}
FIXED_SIMS = {}
for name, table in FIXED_TABLES.items():
    for w in WINDOWS:
        label = f"{name} | static | window={'sentence' if w is None else w}"
        FIXED_SIMS[label] = {s: we.static_similarities(table, fixed_slots[s], w) for s in data}
        print(f'{label:<55}', {s: f'{int(ok.sum())}/{len(ok)}' for s, (_, ok) in FIXED_SIMS[label].items()})

FIXED_META = {'block': 'fixed', 'glove': glove_full.source, 'word2vec': w2v_full.source,
              'n_wic_words': len(WIC_WORDS)}
fixed_rows = we.run_methods(FIXED_SIMS, gold, results_path=RESULTS, meta=FIXED_META)
fixed_common = we.common_mask(FIXED_SIMS)
print('common subset (fixed rows only):', {s: f'{int(m.sum())}/{len(m)}' for s, m in fixed_common.items()})

fixed_df = we.results_frame(fixed_rows).round(3)
fixed_df

10255 vectors of dim 100 read from wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt (81 of the model's words missing)
10102 vectors of dim 300 read from word2vec-google-news-300 (234 of the model's words missing)
fixed | glove-100d full vocab | static | window=0       {'train': '5422/5428', 'dev': '634/638', 'test': '1388/1400'}
fixed | glove-100d full vocab | static | window=1       {'train': '5428/5428', 'dev': '638/638', 'test': '1399/1400'}
fixed | glove-100d full vocab | static | window=sentence {'train': '5428/5428', 'dev': '638/638', 'test': '1399/1400'}
fixed | word2vec-300d full vocab | static | window=0    {'train': '5412/5428', 'dev': '632/638', 'test': '1382/1400'}
fixed | word2vec-300d full vocab | static | window=1    {'train': '5428/5428', 'dev': '638/638', 'test': '1399/1400'}
fixed | word2vec-300d full vocab | static | window=sentence {'train': '5428/5428', 'dev': '638/638', 'test': '1399/1400'}
common subset (fixed rows only): {'train': 

,dev (common),test (common),dev (all),test (all),dev scored,test scored,θ (all)
method,,,,,,,
fixed | glove-100d full vocab | static | window=0,0.528,0.516,0.531,0.515,634/638,1388/1400,0.886
fixed | glove-100d full vocab | static | window=1,0.546,0.518,0.542,0.521,638/638,1399/1400,0.936
fixed | glove-100d full vocab | static | window=sentence,0.549,0.542,0.550,0.550,638/638,1399/1400,0.911
fixed | word2vec-300d full vocab | static | window=0,0.538,0.510,0.542,0.509,632/638,1382/1400,0.678
fixed | word2vec-300d full vocab | static | window=1,0.568,0.559,0.564,0.561,638/638,1399/1400,0.673
fixed | word2vec-300d full vocab | static | window=sentence,0.619,0.581,0.614,0.582,638/638,1399/1400,0.541


## 7. When reporting

- Choose the setting (window, rounds) on dev, not on test.
- Give (common) together with (all) and the coverage table: (common) only holds pairs with
  frequent, in-vocabulary targets.
- word2vec is 300d and was trained on about 100B words; our model saw about 1B.

## 8. Checks (can be skipped)

The fast batched code gives the same vectors as the simple one-at-a-time functions.

In [27]:
# Contextual: same cosine as enc.word_in_context, one sentence at a time.
scores, ok = SIMS['tucker | contextual | rounds=1']['dev']
for k, it in enumerate(data['dev'][:30]):
    a = enc.word_in_context(it.s1, it.char1, rounds=1)
    b = enc.word_in_context(it.s2, it.char2, rounds=1)
    if a is None or b is None:
        assert not ok[k], k
        continue
    assert ok[k] and np.isclose(cosine(a, b), scores[k], atol=1e-5), k
print('contextual_similarities matches word_in_context on dev[:30]')

contextual_similarities matches word_in_context on dev[:30]


In [28]:
# Excluded: same vectors as tk.excluded_role_vector, one window at a time (in-vocabulary targets).
from tensormet.utils import to_np

assert enc.spec.padded
n = enc.spec.order
checked = 0
for pair in slots['dev'][:40]:
    slot = pair[0]
    sentence, i = slot
    if sentence.tokens[i] not in enc.word_index:
        continue
    seq = ['<s>'] * (n - 1) + list(sentence.tokens) + ['</s>']
    p = i + n - 1
    parts = []
    for s0 in range(max(0, p - n + 1), min(p, len(seq) - n) + 1):
        window = tuple(seq[s0:s0 + n])
        if all(t in enc.word_index for t in window):
            v = np.asarray(to_np(tk.excluded_role_vector(window, tk.roles[p - s0])), dtype=float)
            parts.append(v / v.sum())
    vec, has = we.excluded_vectors(enc, [slot])
    assert has[0] == bool(parts)
    if parts:
        assert np.allclose(np.mean(parts, axis=0), vec[0], rtol=1e-3, atol=1e-6)
        checked += 1
print(f'excluded_vectors matches tk.excluded_role_vector on {checked} dev targets')

excluded_vectors matches tk.excluded_role_vector on 26 dev targets
